# Patch · Corregir el precio (PVP en vez de precio con oferta)

**Bug detectado:** el scraping inicial cogía el precio **con oferta** (e.g. 38,25 €) en vez del **PVP tachado** (45,00 €). Para predecir bien necesitamos el PVP como target porque es la señal estable; las ofertas son ruido temporal.

**Cómo lo arregla este notebook:**
1. Hace backup del CSV original.
2. Re-fetchea cada URL.
3. Si hay etiqueta `<s>` con `vc-number` → usa ese precio tachado (PVP), marca `en_oferta=True`.
4. Si no hay `<s>` → el precio actual ES el PVP, marca `en_oferta=False`.
5. Guarda el CSV corregido y añade la columna nueva `en_oferta`.

**Tras correr esto:** vuelve a ejecutar `02_LimpiezaEDA.ipynb` para regenerar los CSVs procesados.

## 1. Setup + backup

In [1]:
import csv
import logging
import random
import re
import shutil
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent
RAW = ROOT / 'data' / 'raw'
CSV = RAW / 'lentiamo_graduadas.csv'
BACKUP = RAW / 'lentiamo_graduadas_BACKUP_pre_patch.csv'
INTERMEDIO = RAW / 'lentiamo_graduadas_intermedio.csv'

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    datefmt='%H:%M:%S', force=True)
log = logging.getLogger('patch')

if not BACKUP.exists():
    shutil.copy(CSV, BACKUP)
    print(f'✅ Backup guardado en: {BACKUP}')
else:
    print(f'ℹ Ya existe un backup en: {BACKUP} (no lo sobrescribo)')

df = pd.read_csv(CSV)
print(f'\nFilas a procesar: {len(df)}')
df.head(2)

ℹ Ya existe un backup en: c:\Users\juan_\Desktop\THE-BRIDGE\RepositorioJuanAMM\Proyecto ML\data\raw\lentiamo_graduadas_BACKUP_pre_patch.csv (no lo sobrescribo)

Filas a procesar: 2875


,url,marca,modelo,tipo,genero,material_montura,forma,tipo_montura,color,talla,ancho_lente,ancho_puente,largo_varilla,calibre_total,peso,polarizadas,precio
0,https://www.lentiamo.es/ray-ban-0rx5698-8109-5...,Ray-Ban,0RX5698 8109 56,graduadas,Unisex,Acetato,Aviador,Montura completa,Marrón,M,56.0,14.0,145.0,133.0,190.0,NaN,112.9
1,https://www.lentiamo.es/saint-laurent-sl-574-0...,Saint Laurent,SL 574 001 52,graduadas,Unisex,Pasta,Rectangulares,Montura completa,Negro,M,52.0,21.0,145.0,139.0,155.0,NaN,249.9


## 2. Funciones de fetch + nuevo extractor de precio

In [2]:
HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/124.0.0.0 Safari/537.36'),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate',
}

DELAY_RANGE = (0.8, 2.0)


def _to_float(val):
    if val is None:
        return None
    s = str(val).replace(',', '.').strip()
    m = re.search(r'-?\d+(?:\.\d+)?', s)
    return float(m.group()) if m else None


def get_session():
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


def fetch(session, url, max_retries=3):
    for i in range(max_retries):
        try:
            r = session.get(url, timeout=20, allow_redirects=True)
            if r.status_code == 200:
                return r.text
            if r.status_code in (403, 429):
                time.sleep(5 * (i + 1))
                continue
            return None
        except requests.RequestException:
            time.sleep(2 * (i + 1))
    return None


def precio_pvp_y_oferta(html):
    """Extrae el PVP (precio tachado si hay oferta, precio actual si no).
    
    Returns: (precio_pvp, en_oferta)
    """
    soup = BeautifulSoup(html, 'html.parser')

    # 1) ¿Hay etiqueta <s> con un span vc-number? → precio tachado = PVP
    for s_tag in soup.find_all('s'):
        span = s_tag.find('span', class_='vc-number')
        if span:
            text = span.get_text(' ', strip=True).replace('\xa0', ' ')
            m = re.search(r'(\d+[,.]\d{2})', text)
            if m:
                return _to_float(m.group(1)), True

    # 2) Sin oferta → el itemprop="price" es el PVP
    meta = soup.find(attrs={'itemprop': 'price'})
    if meta:
        # Preferir 'content' (más fiable que el texto)
        content = meta.get('content')
        if content:
            v = _to_float(content)
            if v:
                return v, False
        text = meta.get_text(' ', strip=True).replace('\xa0', ' ')
        m = re.search(r'(\d+[,.]\d{2})', text)
        if m:
            return _to_float(m.group(1)), False

    # 3) Fallback regex
    m = re.search(r'"price"\s*:\s*"?(\d+[\.,]\d{2})"?', html)
    if m:
        return _to_float(m.group(1)), False

    return None, False


# Test rápido del extractor con HTML simulado
html_oferta = '<s><span class="vc-number">45,00&nbsp;€</span></s><span itemprop="price" content="38.25"></span>'
html_normal = '<span itemprop="price" content="120.00"></span>'
print('Test oferta:    ', precio_pvp_y_oferta(html_oferta))   # debe dar (45.0, True)
print('Test sin oferta:', precio_pvp_y_oferta(html_normal))   # debe dar (120.0, False)

Test oferta:     (45.0, True)
Test sin oferta: (120.0, False)


## 3. Re-fetch incremental

Recorre cada URL y actualiza el precio. Va guardando un CSV intermedio cada 100 filas para que si interrumpes y vuelves, continúe donde estaba.

In [3]:
# Si existe un intermedio, retomamos desde ahí
if INTERMEDIO.exists():
    df_estado = pd.read_csv(INTERMEDIO)
    if 'precio_pvp' in df_estado.columns and len(df_estado) == len(df):
        df = df_estado
        print(f'ℹ Retomando desde intermedio: {df["precio_pvp"].notna().sum()} filas ya hechas')
    else:
        print('ℹ Intermedio incompatible, empiezo desde cero')
        df['precio_pvp'] = pd.NA
        df['en_oferta'] = pd.NA
else:
    df['precio_pvp'] = pd.NA
    df['en_oferta'] = pd.NA

session = get_session()
n_actualizadas = int(df['precio_pvp'].notna().sum())
n_ofertas = int(df.get('en_oferta', pd.Series()).fillna(False).astype(bool).sum())
n_fallos = 0

for i, row in df.iterrows():
    if pd.notna(row['precio_pvp']):
        continue  # ya procesada
    url = row['url']
    html = fetch(session, url)
    time.sleep(random.uniform(*DELAY_RANGE))
    if not html:
        n_fallos += 1
        continue
    pvp, oferta = precio_pvp_y_oferta(html)
    if pvp is None:
        n_fallos += 1
        continue
    df.at[i, 'precio_pvp'] = pvp
    df.at[i, 'en_oferta'] = bool(oferta)
    n_actualizadas += 1
    if oferta:
        n_ofertas += 1

    if n_actualizadas % 50 == 0:
        df.to_csv(INTERMEDIO, index=False)
        log.info('Progreso: %d/%d (en oferta: %d, fallos: %d)',
                 n_actualizadas, len(df), n_ofertas, n_fallos)

df.to_csv(INTERMEDIO, index=False)
print(f'\n✅ Procesado: {n_actualizadas}/{len(df)}')
print(f'   En oferta: {n_ofertas} ({n_ofertas/n_actualizadas*100:.1f}%)')
print(f'   Fallos:    {n_fallos}')

ℹ Retomando desde intermedio: 2854 filas ya hechas

✅ Procesado: 2854/2875
   En oferta: 1257 (44.0%)
   Fallos:    21


## 4. Sustituir precio y limpiar

In [4]:
# Validación: ¿cómo cambia el precio?
comparativa = df[['precio', 'precio_pvp', 'en_oferta']].copy()
comparativa['diff'] = comparativa['precio_pvp'] - comparativa['precio']
comparativa['diff_%'] = (comparativa['diff'] / comparativa['precio'] * 100).round(1)

print('Filas en oferta (PVP > precio actual):')
print(comparativa[comparativa['en_oferta'] == True].head(10))

print(f'\nMediana del descuento (solo ofertas): {comparativa[comparativa["en_oferta"] == True]["diff_%"].median():.1f}%')
print(f'Filas sin precio_pvp (fallos): {comparativa["precio_pvp"].isna().sum()}')

Filas en oferta (PVP > precio actual):
    precio  precio_pvp en_oferta    diff  diff_%
0   112.90        45.0      True  -67.90   -60.1
1   249.90        45.0      True -204.90   -82.0
4   149.90        45.0      True -104.90   -70.0
5    79.99        45.0      True  -34.99   -43.7
9   113.90        45.0      True  -68.90   -60.5
11  115.90        45.0      True  -70.90   -61.2
14  114.90        45.0      True  -69.90   -60.8
15   85.99        45.0      True  -40.99   -47.7
19  239.90        45.0      True -194.90   -81.2
21   79.00        52.0      True  -27.00   -34.2

Mediana del descuento (solo ofertas): -45.8%
Filas sin precio_pvp (fallos): 21


In [5]:
# Para las filas que fallaron, mantener el precio antiguo (no las perdemos)
df['precio_pvp'] = df['precio_pvp'].fillna(df['precio'])
df['en_oferta'] = df['en_oferta'].fillna(False).astype(bool)

# Sustituir el precio por el PVP correcto
df['precio'] = df['precio_pvp']
df = df.drop(columns=['precio_pvp'])

# Mover en_oferta antes del precio para que sea más visible
cols = [c for c in df.columns if c != 'precio'] + ['precio']
df = df[cols]

df.head(3)

,url,marca,modelo,tipo,genero,material_montura,forma,tipo_montura,color,talla,ancho_lente,ancho_puente,largo_varilla,calibre_total,peso,polarizadas,en_oferta,precio
0,https://www.lentiamo.es/ray-ban-0rx5698-8109-5...,Ray-Ban,0RX5698 8109 56,graduadas,Unisex,Acetato,Aviador,Montura completa,Marrón,M,56.0,14.0,145.0,133.0,190.0,NaN,True,45.0
1,https://www.lentiamo.es/saint-laurent-sl-574-0...,Saint Laurent,SL 574 001 52,graduadas,Unisex,Pasta,Rectangulares,Montura completa,Negro,M,52.0,21.0,145.0,139.0,155.0,NaN,True,45.0
2,https://www.lentiamo.es/saint-laurent-sl-m138-...,Saint Laurent,SL M138 001 55,graduadas,Mujer,Pasta,Cat Eye,Montura completa,Negro,M,55.0,18.0,145.0,138.0,160.0,NaN,False,249.9


In [8]:
# Sobrescribir el CSV original (el backup ya está hecho en data/raw/lentiamo_graduadas_BACKUP_pre_patch.csv)
df.to_csv(CSV, index=False)
print(f'✅ {CSV} actualizado.')
print(f'   - {len(df)} filas')
print(f'   - {df["en_oferta"].sum()} en oferta')
print(f'   - precio mediano (PVP): {df["precio"].median():.2f} €')

# Borrar el intermedio porque ya hemos terminado
if INTERMEDIO.exists():
    INTERMEDIO.unlink()
    print(f'   - intermedio borrado')

✅ c:\Users\juan_\Desktop\THE-BRIDGE\RepositorioJuanAMM\Proyecto ML\data\raw\lentiamo_graduadas.csv actualizado.
   - 2875 filas
   - 1257 en oferta
   - precio mediano (PVP): 99.90 €
   - intermedio borrado


## 5. Siguientes pasos

1. **Vuelve a ejecutar `02_LimpiezaEDA.ipynb`** entero — usa el CSV ya corregido y regenera `data/processed/lentiamo_graduadas_clean.csv` + splits.
2. La nueva columna `en_oferta` aparecerá automáticamente en el dataset enriquecido. En el `02` puedes añadirla a `feats_num` (es 0/1) si quieres que el modelo aprenda si el descuento aporta info.
3. Si algo va mal, restaura desde `data/raw/lentiamo_graduadas_BACKUP_pre_patch.csv`.